In [2]:
import os

os.listdir()

['.ipynb_checkpoints',
 'ai_gpu_related_services.csv',
 'ai_readiness_index.ipynb',
 'ai_type_summary.csv',
 'benchmark_pricing_summary.csv',
 'gpu_ai_infrastructure.ipynb',
 'Infrastructure_Dataset.ipynb',
 'native_ai_services_enriched.csv',
 'native_cloud_service_catalog.csv',
 'native_service_category_summary.csv',
 'normalized_vm_pricing.csv',
 'other_marketplace_service_catalog.csv',
 'Pricing Data Analysis.ipynb',
 'provider_ai_service_share.csv',
 'Service_Catalog.ipynb',
 'service_category_pivot_summary.csv',
 'service_category_summary.csv']

In [3]:
import pandas as pd
import numpy as np

native_service_catalog = pd.read_csv("native_cloud_service_catalog.csv")
native_ai_services = pd.read_csv("native_ai_services_enriched.csv")
pricing_df = pd.read_csv("normalized_vm_pricing.csv")
benchmark_pricing = pd.read_csv("benchmark_pricing_summary.csv")
service_category_summary = pd.read_csv("native_service_category_summary.csv")

In [4]:
print("native_service_catalog")
print(native_service_catalog.columns)

print("\nnative_ai_services")
print(native_ai_services.columns)

print("\npricing_df")
print(pricing_df.columns)

print("\nbenchmark_pricing")
print(benchmark_pricing.columns)

print("\nservice_category_summary")
print(service_category_summary.columns)

native_service_catalog
Index(['provider', 'service_code', 'service_name', 'service_title', 'category',
       'source', 'collection_date', 'category_v2', 'category_v3'],
      dtype='object')

native_ai_services
Index(['provider', 'service_code', 'service_name', 'service_title', 'category',
       'source', 'collection_date', 'category_v2', 'category_v3',
       'service_origin', 'ai_type'],
      dtype='object')

pricing_df
Index(['provider', 'benchmark_type', 'category', 'instance_type', 'region',
       'vcpu', 'memory_gb', 'hourly_price_usd', 'pricing_method',
       'cpu_component_price', 'ram_component_price'],
      dtype='object')

benchmark_pricing
Index(['provider', 'benchmark_type', 'category', 'avg_hourly_price',
       'median_hourly_price', 'avg_monthly_cost', 'avg_price_per_vcpu',
       'avg_price_per_gb_ram'],
      dtype='object')

service_category_summary
Index(['provider', 'category', 'service_count'], dtype='object')


In [5]:
ai_count = (
    native_ai_services
    .groupby("provider")
    .size()
    .reset_index(name="ai_service_count")
)

ai_count

,provider,ai_service_count
0,AWS,6
1,Azure,4
2,GCP,7


In [6]:
ai_diversity = (
    native_ai_services
    .groupby("provider")["ai_type"]
    .nunique()
    .reset_index(name="ai_type_diversity")
)

ai_diversity

,provider,ai_type_diversity
0,AWS,3
1,Azure,3
2,GCP,2


In [7]:
infra_breadth = (
    native_service_catalog
    .groupby("provider")["category"]
    .nunique()
    .reset_index(name="infra_breadth")
)

infra_breadth

,provider,infra_breadth
0,AWS,11
1,Azure,8
2,GCP,11


In [8]:
total_services = (
    native_service_catalog
    .groupby("provider")
    .size()
    .reset_index(name="total_services")
)

ai_share = (
    ai_count.merge(
        total_services,
        on="provider"
    )
)

ai_share["ai_share_pct"] = (
    ai_share["ai_service_count"]
    / ai_share["total_services"]
) * 100

ai_share

,provider,ai_service_count,total_services,ai_share_pct
0,AWS,6,69,8.695652
1,Azure,4,32,12.500000
2,GCP,7,359,1.949861


In [9]:
pricing_score = (
    benchmark_pricing
    .groupby("provider")
    ["avg_hourly_price"]
    .mean()
    .reset_index()
)

pricing_score

,provider,avg_hourly_price
0,AWS,0.205792
1,GCP,0.054459


In [10]:
readiness = (
    ai_count
    .merge(ai_diversity,on="provider")
    .merge(infra_breadth,on="provider")
    .merge(
        ai_share[[
            "provider",
            "ai_share_pct"
        ]],
        on="provider"
    )
    .merge(
        pricing_score,
        on="provider"
    )
)

readiness

,provider,ai_service_count,ai_type_diversity,infra_breadth,ai_share_pct,avg_hourly_price
0,AWS,6,3,11,8.695652,0.205792
1,GCP,7,2,11,1.949861,0.054459


In [11]:
pricing_df["provider"].value_counts()

provider
AWS      78
GCP      15
Azure     4
Name: count, dtype: int64

In [12]:
pricing_df[
    pricing_df["provider"]=="Azure"
]

,provider,benchmark_type,category,instance_type,region,vcpu,memory_gb,hourly_price_usd,pricing_method,cpu_component_price,ram_component_price
78,Azure,General Purpose Small,General Purpose,Standard_D2s_v7,eastus,2,8,0.132,Official instance price,NaN,NaN
79,Azure,General Purpose Small,General Purpose,Standard_D2s_v7,westus2,2,8,0.132,Official instance price,NaN,NaN
80,Azure,Memory Optimized Small,Memory Optimized,Standard_E2s_v7,eastus,2,16,0.174,Official instance price,NaN,NaN
81,Azure,Memory Optimized Small,Memory Optimized,Standard_E2s_v7,westus2,2,16,0.174,Official instance price,NaN,NaN


In [13]:
pricing_score = (
    pricing_df
    .groupby("provider")
    ["hourly_price_usd"]
    .mean()
    .reset_index()
)

pricing_score

,provider,hourly_price_usd
0,AWS,0.226797
1,Azure,0.153000
2,GCP,0.054459


In [14]:
readiness

,provider,ai_service_count,ai_type_diversity,infra_breadth,ai_share_pct,avg_hourly_price
0,AWS,6,3,11,8.695652,0.205792
1,GCP,7,2,11,1.949861,0.054459


In [15]:
print(ai_count)
print()
print(ai_diversity)
print()
print(infra_breadth)
print()
print(ai_share)

  provider  ai_service_count
0      AWS                 6
1    Azure                 4
2      GCP                 7

  provider  ai_type_diversity
0      AWS                  3
1    Azure                  3
2      GCP                  2

  provider  infra_breadth
0      AWS             11
1    Azure              8
2      GCP             11

  provider  ai_service_count  total_services  ai_share_pct
0      AWS                 6              69      8.695652
1    Azure                 4              32     12.500000
2      GCP                 7             359      1.949861


In [16]:
pricing_score = (
    pricing_df
    .groupby("provider")["hourly_price_usd"]
    .mean()
    .reset_index(name="avg_hourly_price")
)

readiness = (
    ai_count
    .merge(ai_diversity, on="provider", how="outer")
    .merge(infra_breadth, on="provider", how="outer")
    .merge(
        ai_share[["provider", "ai_share_pct"]],
        on="provider",
        how="outer"
    )
    .merge(
        pricing_score,
        on="provider",
        how="outer"
    )
)

readiness

,provider,ai_service_count,ai_type_diversity,infra_breadth,ai_share_pct,avg_hourly_price
0,AWS,6,3,11,8.695652,0.226797
1,Azure,4,3,8,12.500000,0.153000
2,GCP,7,2,11,1.949861,0.054459


In [17]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

readiness["ai_service_score"] = scaler.fit_transform(
    readiness[["ai_service_count"]]
)

readiness["diversity_score"] = scaler.fit_transform(
    readiness[["ai_type_diversity"]]
)

readiness["infra_score"] = scaler.fit_transform(
    readiness[["infra_breadth"]]
)

readiness["share_score"] = scaler.fit_transform(
    readiness[["ai_share_pct"]]
)

# price 越低越好，所以要反轉
readiness["pricing_score"] = 1 - scaler.fit_transform(
    readiness[["avg_hourly_price"]]
)

In [18]:
readiness["ai_readiness_index"] = (
      readiness["ai_service_score"] * 0.25
    + readiness["diversity_score"] * 0.20
    + readiness["infra_score"] * 0.20
    + readiness["share_score"] * 0.20
    + readiness["pricing_score"] * 0.15
)

In [19]:
readiness_ranked = readiness.sort_values(
    "ai_readiness_index",
    ascending=False
)

readiness_ranked[[
    "provider",
    "ai_readiness_index",
    "ai_service_count",
    "ai_type_diversity",
    "infra_breadth",
    "ai_share_pct",
    "avg_hourly_price"
]]

,provider,ai_readiness_index,ai_service_count,ai_type_diversity,infra_breadth,ai_share_pct,avg_hourly_price
0,AWS,0.694547,6,3,11,8.695652,0.226797
2,GCP,0.600000,7,2,11,1.949861,0.054459
1,Azure,0.464232,4,3,8,12.500000,0.153000


In [20]:
readiness_ranked.to_csv(
    "ai_readiness_index_v1.csv",
    index=False
)